In [3]:
import import_ipynb
from rco_test import *
from ast import *
from utils import *
from x86_ast import *


In [4]:
def select_arg(e: expr) -> arg:
        # YOUR CODE HERE
        match e:
            case Constant(value):
                return Immediate(value)
            case Name(var):
                return Variable(var)
         

In [5]:
def select_stmt(s: stmt) -> list[instr]:
        # YOUR CODE HERE
        match s:
          
            case Assign([Name(var)],UnaryOp(USub(), v)):
                val = select_arg(v)
                return [Instr('movq',[val, Reg('rax')]),
                        Instr('negq',[Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left,Add(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('addq',[r,Variable(var)])]
                elif isinstance(right, Name) and right.id == var:
                     return [Instr('addq',[l,Variable(var)])]
                else:
                    return [Instr('movq',[l, Reg('rax')]),
                        Instr('addq',[r,Reg('rax')]),
                        Instr('movq',[Reg('rax'), Variable(var)])]
            case Assign([Name(var)], BinOp(left, Sub(), right)):
                l = select_arg(left)
                r = select_arg(right)
                if isinstance(left, Name) and left.id == var:
                     return [Instr('subq',[r,Variable(var)])]
                elif isinstance(right, Name) and right.id == var:
                     return [Instr('subq', [l, Variable(var)])]
                else:
                     return [Instr('movq',[l,Reg('rax')]),
                        Instr('subq',[r,Reg('rax')]),
                        Instr('movq', [Reg('rax'),Variable(var)])]
            case Assign([Name(var)], Call(Name('input_int'))):
                return [Callq('read_int',1),
                        Instr('movq', [Reg('rax'), Variable(var)])]
            case Assign([Name(var)],value):
                  new_value = select_arg(value)
                  return [Instr('movq',[new_value,Variable(var)])]
            case Expr(Call(Name('print'),[arg])):
                new_arg = select_arg(arg)
                return [Instr('movq',[new_arg, Reg('rdi')]), 
                        Callq('print_int',1)]
           

In [6]:
def select_instruction(p : Module) -> X86Program:
    match p:
        case Module(body):
            new_body = []
            for stmt in body:
                new_body.extend(select_stmt(stmt))
                
            return X86Program(new_body)

In [7]:
if __name__ == "__main__":
    import textwrap
    code = textwrap.dedent("""
    x = (5 + 3) + (-2 + 3)
    y = (x + 1) + (10 - x)
    print(y + 3)""")
    parsed_code = parse(code)
    rco_code = remove_complex_operands(parsed_code)
    select_instr_program = select_instruction(rco_code)
    print(select_instr_program)
    

	.globl main
main:
    movq $5, %rax
    addq $3, %rax
    movq %rax, temp.0
    movq $2, %rax
    negq %rax
    movq %rax, temp.1
    movq temp.1, %rax
    addq $3, %rax
    movq %rax, temp.2
    movq temp.0, %rax
    addq temp.2, %rax
    movq %rax, x
    movq x, %rax
    addq $1, %rax
    movq %rax, temp.3
    movq $10, %rax
    subq x, %rax
    movq %rax, temp.4
    movq temp.3, %rax
    addq temp.4, %rax
    movq %rax, y
    movq y, %rax
    addq $3, %rax
    movq %rax, temp.5
    movq temp.5, %rdi
    callq print_int


